In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
# RetrievalQA is deprecated in newer LangChain versions
# Use create_retrieval_chain or LCEL pattern instead
# from langchain.chains import RetrievalQA  # Deprecated
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline
import os
import warnings
warnings.filterwarnings("ignore")

/Users/taha/Projects/langchain-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define the path to the specific TSV file
dataset_path = "data/amazon_reviews_us_Electronics_v1_00.tsv"

# Verify the file exists
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"File not found at {dataset_path}. Please ensure the dataset is correctly attached.")

# Load the TSV dataset (using tab separator)
df = pd.read_csv(dataset_path, sep='\t', on_bad_lines='skip')  # 'skip' handles malformed lines

# Select relevant columns
df = df[['product_id', 'product_title', 'star_rating', 'review_body']].dropna()

# Add mock price since it's not in the dataset
df['price'] = np.random.randint(50, 1500, size=len(df))

# Use a subset for EDA and processing (e.g., 1000 rows)
df = df.head(1000)
print("Dataset shape:", df.shape)

Dataset shape: (1000, 5)


In [ ]:
# Basic info
print("Dataset Info:")
print(df.info())

# Summary statistics
print("\nSummary Statistics:")
print(df.describe())

# 1. Distribution of Star Ratings
plt.figure(figsize=(8, 5))
sns.countplot(x='star_rating', data=df, palette='viridis')
plt.title('Distribution of Star Ratings')
plt.xlabel('Star Rating')
plt.ylabel('Count')
plt.show()

# 2. Distribution of Mock Prices
plt.figure(figsize=(8, 5))
sns.histplot(df['price'], bins=20, kde=True, color='blue')
plt.title('Distribution of Mock Prices')
plt.xlabel('Price ($)')
plt.ylabel('Frequency')
plt.show()

# 3. Review Length Analysis
df['review_length'] = df['review_body'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(8, 5))
sns.histplot(df['review_length'], bins=30, kde=True, color='green')
plt.title('Distribution of Review Lengths (Word Count)')
plt.xlabel('Review Length (Words)')
plt.ylabel('Frequency')
plt.show()

# 4. Top 10 Most Reviewed Products
top_products = df['product_title'].value_counts().head(10)
plt.figure(figsize=(10, 6))
top_products.plot(kind='bar', color='purple')
plt.title('Top 10 Most Reviewed Products')
plt.xlabel('Product Title')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=45, ha='right')
plt.show()

# 5. Missing Values
print("\nMissing Values:")
print(df.isnull().sum())